# AI Tutor — Document Parsing Benchmark

Run these cells in order. Upload one ZIP containing the 11 PDFs when prompted.

In [ ]:
# 1) Clone or update the repository
%cd /content
!git clone https://github.com/basmala-19/AI-Tutor-EDU-.git || true
%cd /content/AI-Tutor-EDU-
!git pull origin main
!git log --oneline -1

In [ ]:
# 2) Install OS and Python dependencies
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng
!pip -q install -r requirements-colab.txt
!tesseract --list-langs
!PYTHONPATH=. python -c "import docling; print('Docling:', docling.__version__)"

In [ ]:
# 3) Upload ONE ZIP that contains your PDF books (not nested in another ZIP).
from google.colab import files
uploaded = files.upload()
assert uploaded, 'Upload a ZIP file containing the PDF books.'
zip_name = next(iter(uploaded))
print('Uploaded:', zip_name)

In [ ]:
# 4) Extract the books and verify the expected PDFs.
from pathlib import Path
import shutil

books_dir = Path('/content/books')
shutil.rmtree(books_dir, ignore_errors=True)
books_dir.mkdir()
shutil.unpack_archive(zip_name, books_dir)
pdfs = sorted(books_dir.rglob('*.pdf'))
print(f'Found {len(pdfs)} PDFs:')
for pdf in pdfs:
    print(' -', pdf)
assert pdfs, 'No PDFs were found. ZIP the PDF files themselves and upload again.'

In [ ]:
# 5) Optional LlamaParse recovery. Leave blank to use only Docling/Tesseract.
import os
from getpass import getpass
use_llama = False  # Change to True only if you want cloud fallback.
if use_llama:
    os.environ['LLAMA_CLOUD_API_KEY'] = getpass('LlamaCloud API key: ')

In [ ]:
# 6) Fast diagnostic for Math_EN. This is the correct runner; it records
# parser attempts and a traceback if the pipeline itself fails.
import subprocess

math_files = list(books_dir.rglob('Math_EN_prim1_Tr2.pdf'))
if math_files:
    command = [
        'python', '-m', 'benchmark.quality_benchmark', str(books_dir),
        '--files', str(math_files[0]),
        '--output', 'benchmark/reports/math_en_diagnostic.csv',
    ]
    result = subprocess.run(command, env={**os.environ, 'PYTHONPATH': '.'}, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
else:
    print('Math_EN_prim1_Tr2.pdf was not found; skipped.')

In [ ]:
# 7) Inspect the Math_EN result. Do not use benchmark/evaluator.py for this diagnostic.
import pandas as pd
from pathlib import Path

result_path = Path('benchmark/reports/math_en_diagnostic.csv')
if result_path.exists():
    math_result = pd.read_csv(result_path)
    display(math_result.T)
    if 'parser_attempts' in math_result.columns:
        print(math_result.loc[0, 'parser_attempts'])
    if 'traceback' in math_result.columns and pd.notna(math_result.loc[0, 'traceback']):
        print(math_result.loc[0, 'traceback'])

In [ ]:
# 8) Full quality benchmark. This can take a long time with Docling.
# It creates an after.csv with parser attempts and all eight quality metrics.
!PYTHONPATH=. python -m benchmark.quality_benchmark /content/books --output benchmark/reports/after.csv --artifacts-dir benchmark/artifacts

In [ ]:
# 9) Display EVERY result saved in after.csv.
import pandas as pd

results = pd.read_csv('benchmark/reports/after.csv')
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', None)
display(results)

print('Status counts:')
display(results['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='files'))

# Failed files keep the reason and full traceback in the CSV.
issues = results[results['status'].isin(['FAIL', 'ERROR'])]
if not issues.empty:
    print('Failures / errors:')
    display(issues)

print('Saved CSV:', '/content/AI-Tutor-EDU-/benchmark/reports/after.csv')
print('Saved JSON:', '/content/AI-Tutor-EDU-/benchmark/artifacts/json/')
print('Saved Markdown:', '/content/AI-Tutor-EDU-/benchmark/artifacts/markdown/')

In [ ]:
# 10) Download the CSV and all per-book JSON/Markdown artifacts.
from google.colab import files
import shutil
shutil.make_archive('/content/ai_tutor_parsed_outputs', 'zip', 'benchmark/artifacts')
files.download('benchmark/reports/after.csv')
files.download('/content/ai_tutor_parsed_outputs.zip')